# Reject option su ResNet18 v3 - softmax confidence

Valuta il checkpoint migliore del run `resnet18_exnovo_v3_20260629-2107` introducendo una reject option basata su `max softmax probability`.

Il modello non viene riallenato: si misura il trade-off tra coverage e accuratezza sui campioni accettati.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import datetime
import subprocess
from getpass import getpass
from pathlib import Path

REPO_OWNER = 'fabriziodrr'
REPO_NAME = 'ProjectML'
BASE_BRANCH = 'exp/reject-softmax-confidence-v3'
RUN_ID = datetime.datetime.now().strftime('%Y%m%d-%H%M')
RUN_BRANCH = f'exp-reject-softmax-confidence-v3-{RUN_ID}'
RUN_NAME = f'reject_softmax_confidence_v3_{RUN_ID}'
REPO_PATH = f'/content/{REPO_NAME}'
PUBLIC_REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

def sanitize(text):
    if 'AUTH_REPO_URL' in globals():
        text = text.replace(AUTH_REPO_URL, 'https://***@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git')
    if 'GITHUB_TOKEN' in globals() and GITHUB_TOKEN:
        text = text.replace(GITHUB_TOKEN, '***')
    return text

def run(cmd, cwd=None, check=True):
    printable = sanitize(' '.join(cmd))
    print('$', printable, flush=True)
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(sanitize(result.stdout), flush=True)
    if result.stderr:
        print(sanitize(result.stderr), flush=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {printable}')
    return result

GITHUB_TOKEN = getpass('GitHub token personale con permessi repo: ')
AUTH_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if not os.path.exists(REPO_PATH):
    run(['git', 'clone', '--branch', BASE_BRANCH, AUTH_REPO_URL, REPO_PATH])
else:
    print('Repo gia presente nel runtime.')

run(['git', 'remote', 'set-url', 'origin', AUTH_REPO_URL], cwd=REPO_PATH)
run(['git', 'pull', '--rebase', 'origin', BASE_BRANCH], cwd=REPO_PATH)
run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH)
run(['git', 'switch', '-c', RUN_BRANCH], cwd=REPO_PATH)
os.chdir(REPO_PATH)
print('Run branch:', RUN_BRANCH)
print('Run name:', RUN_NAME)

In [ ]:
import json
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import matplotlib.pyplot as plt

EXP_NAME = 'reject_softmax_confidence_v3'
REJECT_METHOD = 'softmax_confidence'
SEED = 42
NUM_CLASSES = 8
BATCH_SIZE = 64
NUM_WORKERS = 0
VAL_SIZE = 0.20
DROPOUT_P = 0.25

DRIVE_ZIP_PATH = '/content/drive/MyDrive/waste_type_identification.zip'
DRIVE_DATASET_DIR = '/content/drive/MyDrive/waste_type_identification'
LOCAL_DATASET_DIR = '/content/waste_type_identification'
CHECKPOINT_PATH = '/content/drive/MyDrive/ProjectML_checkpoints/resnet18_exnovo_v3_20260629-2107/resnet18_exnovo_v3_best.pth'

RESULTS_DIR = Path(REPO_PATH) / 'results' / RUN_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['Battery', 'Clothing', 'Glass', 'Metal', 'Organic', 'Papery', 'Plastic', 'Undifferentiated']
LABEL_MAP = {
    'battery': 0,
    'clothes': 1, 'shoes': 1,
    'brown': 2, 'green': 2, 'transparent': 2,
    'metal': 3,
    'organic': 4,
    'cardboard': 5, 'paper': 5,
    'plastic': 6,
    'undifferentiated': 7,
}
IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Checkpoint:', CHECKPOINT_PATH)

In [ ]:
local_dataset = Path(LOCAL_DATASET_DIR)
drive_dataset = Path(DRIVE_DATASET_DIR)
drive_zip = Path(DRIVE_ZIP_PATH)

if not local_dataset.exists():
    if drive_dataset.exists():
        shutil.copytree(drive_dataset, local_dataset)
    elif drive_zip.exists():
        local_dataset.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(drive_zip, 'r') as zf:
            zf.extractall(local_dataset)
    else:
        raise FileNotFoundError(f'Non trovo ne {drive_dataset} ne {drive_zip}')

samples = []
for dirpath, dirnames, filenames in os.walk(local_dataset, followlinks=False):
    folder = Path(dirpath).name.lower()
    if folder not in LABEL_MAP:
        continue
    label = LABEL_MAP[folder]
    for fname in filenames:
        if fname.lower().endswith(IMG_EXTENSIONS):
            path = Path(dirpath) / fname
            samples.append({'path': str(path), 'relative_path': str(path.relative_to(local_dataset)), 'folder': folder, 'label': label, 'class_name': CLASS_NAMES[label]})

df = pd.DataFrame(samples).sort_values('relative_path').reset_index(drop=True)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
_, val_idx = next(splitter.split(df['relative_path'], df['label']))
df['split'] = 'train'
df.loc[val_idx, 'split'] = 'val'
val_df = df[df['split'] == 'val'].reset_index(drop=True)
df[['relative_path', 'folder', 'label', 'class_name', 'split']].to_csv(RESULTS_DIR / 'split.csv', index=False)
print('Validation samples:', len(val_df))
print(val_df['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0))

In [ ]:
class WasteDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['label']), row['relative_path']

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_loader = DataLoader(WasteDataset(val_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Sequential(nn.Dropout(p=DROPOUT_P), nn.Linear(num_features, NUM_CLASSES))
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

all_probs, all_targets, all_paths = [], [], []
with torch.no_grad():
    for images, targets, paths in val_loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets.numpy())
        all_paths.extend(paths)

probs = np.concatenate(all_probs)
y_true = np.concatenate(all_targets)
y_pred = probs.argmax(axis=1)
sorted_probs = np.sort(probs, axis=1)
confidence = probs.max(axis=1)
margin = sorted_probs[:, -1] - sorted_probs[:, -2]
score = confidence

predictions_df = pd.DataFrame({
    'relative_path': all_paths,
    'true_label': y_true,
    'true_class': [CLASS_NAMES[i] for i in y_true],
    'pred_label': y_pred,
    'pred_class': [CLASS_NAMES[i] for i in y_pred],
    'confidence': confidence,
    'top2_margin': margin,
    'correct': y_true == y_pred,
})
predictions_df.to_csv(RESULTS_DIR / 'predictions.csv', index=False)
print('Base accuracy:', float((y_true == y_pred).mean()))
print('Base balanced accuracy:', float(balanced_accuracy_score(y_true, y_pred)))

In [ ]:
thresholds = np.array([0.0, 0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99])
rows = []
for threshold in thresholds:
    accepted = score >= threshold
    n_accepted = int(accepted.sum())
    coverage = float(n_accepted / len(y_true))
    if n_accepted > 0:
        acc = float((y_true[accepted] == y_pred[accepted]).mean())
        bal_acc = float(balanced_accuracy_score(y_true[accepted], y_pred[accepted]))
        errors = int((y_true[accepted] != y_pred[accepted]).sum())
    else:
        acc = np.nan
        bal_acc = np.nan
        errors = 0
    rows.append({'threshold': threshold, 'coverage': coverage, 'rejection_rate': 1.0 - coverage, 'accepted': n_accepted, 'rejected': int(len(y_true) - n_accepted), 'accepted_accuracy': acc, 'accepted_balanced_accuracy': bal_acc, 'accepted_errors': errors})
curve_df = pd.DataFrame(rows)
curve_df.to_csv(RESULTS_DIR / 'rejection_curve.csv', index=False)
print(curve_df)

eligible = curve_df[curve_df['coverage'] >= 0.95].copy()
if eligible.empty:
    selected = curve_df.iloc[curve_df['accepted_balanced_accuracy'].idxmax()]
else:
    selected = eligible.iloc[eligible['accepted_balanced_accuracy'].idxmax()]
selected_threshold = float(selected['threshold'])
accepted = score >= selected_threshold

report = classification_report(y_true[accepted], y_pred[accepted], labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES, zero_division=0)
cm = confusion_matrix(y_true[accepted], y_pred[accepted], labels=list(range(NUM_CLASSES)))
with open(RESULTS_DIR / 'classification_report_accepted.txt', 'w') as f:
    f.write(report)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_DIR / 'confusion_matrix_accepted.csv')
print('Selected threshold:', selected_threshold)
print(report)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(curve_df['coverage'], curve_df['accepted_accuracy'], marker='o', label='accepted accuracy')
plt.plot(curve_df['coverage'], curve_df['accepted_balanced_accuracy'], marker='o', label='accepted balanced accuracy')
plt.gca().invert_xaxis()
plt.xlabel('Coverage')
plt.ylabel('Metric on accepted samples')
plt.title('Reject option - softmax confidence')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'rejection_curve.png', dpi=160)
plt.show()

summary = {
    'run_name': RUN_NAME,
    'run_branch': RUN_BRANCH,
    'source_model': 'resnet18_exnovo_v3_20260629-2107',
    'reject_method': REJECT_METHOD,
    'base_accuracy': float((y_true == y_pred).mean()),
    'base_balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    'selected_threshold': selected_threshold,
    'selected_coverage': float(selected['coverage']),
    'selected_rejection_rate': float(selected['rejection_rate']),
    'selected_accepted_accuracy': float(selected['accepted_accuracy']),
    'selected_accepted_balanced_accuracy': float(selected['accepted_balanced_accuracy']),
    'selection_rule': 'maximize accepted balanced accuracy with coverage >= 0.95',
}
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
config = {'experiment': EXP_NAME, 'reject_method': REJECT_METHOD, 'checkpoint_path': CHECKPOINT_PATH, 'thresholds': thresholds.tolist(), 'selection_rule': summary['selection_rule']}
with open(RESULTS_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
os.chdir(REPO_PATH)
run(['git', 'config', 'user.name', 'fabriziodrr'], cwd=REPO_PATH)
run(['git', 'config', 'user.email', 'fabriziodrr@users.noreply.github.com'], cwd=REPO_PATH)
run(['git', 'add', str(RESULTS_DIR.relative_to(REPO_PATH)), 'src/reject_option/reject_softmax_confidence_v3_colab.ipynb'], cwd=REPO_PATH)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_PATH).decode().strip()
if not status:
    print('Nessuna modifica da committare.')
else:
    run(['git', 'commit', '-m', f'Add reject softmax confidence v3 results {RUN_ID}'], cwd=REPO_PATH)
    auth_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    try:
        run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=REPO_PATH)
        run(['git', 'push', '-u', 'origin', RUN_BRANCH], cwd=REPO_PATH)
    finally:
        run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH, check=False)
print('Risultati reject option sincronizzati su GitHub.')